# 1/7 - B. The pressure signal
Who presses the ball carrier, how covered the passing lanes are, the unified team-pressure flag and press events, stoppages, total pressing time.

*Original sections: sections 0, 0b, 1, 2, 3, 4.*  Barca (team 0) vs Atletico (team 1), first half.

**Run the notebooks in order** (each one reads what the earlier ones saved in `new_cache/pressure_analysis/state/`):

| # | notebook | what it covers |
|---|---|---|
| 1 | **01_pressure_signal.ipynb** | B. The pressure signal |
| 2 | 02_where_zone_channel_line.ipynb | C. WHERE - zone, channel, defensive line, shape |
| 3 | 03_timing_and_possession.ipynb | D. WHEN - reaction speed and possession-level pressure |
| 4 | 04_passes_outcomes_triggers.ipynb | E/F. EFFECT and TRIGGERS |
| 5 | 05_summary_and_registry.ipynb | G/H. Summary + event registry |
| 6 | 06_clips.ipynb | H. Clips |
| 7 | 07_exports.ipynb | I. Exports |

Shared setup (paths, tables, data contract, team-label orientation, helpers) lives in `pressure_common.py` - keep it in the same folder as the notebooks.

In [1]:
from pressure_common import *

player table: player_frame_table_named.parquet (1593443, 41)
ball table  : ball_frame_table_named.parquet (70327, 22)
column guard: dropped from the player table (the ball table is the source for these): ['carrier_track_id', 'is_stoppage', 'possession_team']
ORIENTATION: table had team 0 attacking -x / team 1 attacking +x -> team ids swapped on load (now team 0 attacks +x, team 1 attacks -x, as the analysis cells assume)
TEAM_NAMES: {0: 'Barca', 1: 'Atletico'}
ball-table carrier_team agrees with the player table's team ids on 100.0% of carrier frames

roles: {'player': 1351246, 'referee': 179020, 'goalkeeper': 63177}
frames 0..70325 | ball rows 70327 | carrier coverage 48.8% | goalkeepers [1, 2]
data contract OK


## 0. Team defensive baseline
Average depth of each team's back line (last-3 defenders), computed once up front, restricted to frames where that team is out of possession (their actual defending shape, not inflated by attacking-phase positioning). Used in section 1 to tell "stepped out to press" apart from "block was already sitting there".

In [2]:
LINE_HEIGHT_DEFENDERS_N = 3

_poss_for_baseline = ball_frame_table[["frame_idx", "possession_team"]]

_outfield_baseline = player_frame_table[player_frame_table["role"] == "player"].dropna(subset=["pitch_x", "pitch_y"]).copy()
_outfield_baseline = _outfield_baseline.merge(_poss_for_baseline, on="frame_idx", how="left")
_outfield_baseline = _outfield_baseline[
    _outfield_baseline["possession_team"].notna() &
    (_outfield_baseline["team"] != _outfield_baseline["possession_team"])
]  # keep only frames where this team is out of possession -- their real defensive shape

_outfield_baseline["dist_from_own_goal"] = np.where(
    _outfield_baseline["team"] == 0,
    _outfield_baseline["pitch_x"],
    PITCH_LENGTH - _outfield_baseline["pitch_x"]
)
_outfield_baseline["dist_from_own_goal"] = _outfield_baseline["dist_from_own_goal"].clip(0, PITCH_LENGTH)

team_defensive_line_avg = (
    _outfield_baseline.sort_values("dist_from_own_goal")
    .groupby(["frame_idx", "team"])
    .head(LINE_HEIGHT_DEFENDERS_N)
    .groupby("team")["dist_from_own_goal"]
    .mean()
    .to_dict()
)
print("team average defensive line depth, out-of-possession only (last-3 defenders, m from own goal):", team_defensive_line_avg)

team average defensive line depth, out-of-possession only (last-3 defenders, m from own goal): {0.0: 45.34747478395824, 1.0: 27.474277093622717}


## 0b. Stoppage detection
A no-carrier gap in `carrier_track_id` lasting >= `STOPPAGE_GAP_SEC` (10s) is flagged as a real stoppage (injury, goal, VAR check, cooling break) rather than ordinary loose-ball noise -- verified by eye against the video. `stoppage_frame_lookup` is a per-frame boolean used downstream (sections 4, 12, 14) so dead time doesn't get silently counted as live out-of-possession play.

In [3]:
STOPPAGE_GAP_SEC = 10.0
STOPPAGE_GAP_FRAMES = int(STOPPAGE_GAP_SEC * FPS)

carrier_missing = ball_frame_table[["frame_idx", "carrier_track_id"]].sort_values("frame_idx").reset_index(drop=True)
carrier_missing["is_missing"] = carrier_missing["carrier_track_id"].isna()
carrier_missing["gap_id"] = (carrier_missing["is_missing"] != carrier_missing["is_missing"].shift()).cumsum()

gap_runs = (
    carrier_missing[carrier_missing["is_missing"]]
    .groupby("gap_id")
    .agg(start_frame=("frame_idx", "first"), end_frame=("frame_idx", "last"), n_frames=("frame_idx", "count"))
    .reset_index(drop=True)
)
gap_runs["duration_sec"] = gap_runs["n_frames"] / FPS

stoppages = gap_runs[gap_runs["duration_sec"] >= STOPPAGE_GAP_SEC].reset_index(drop=True)

# vectorized frame -> is_stoppage lookup
_stoppage_starts = stoppages["start_frame"].values
_stoppage_ends = stoppages["end_frame"].values

all_frames = ball_frame_table[["frame_idx"]].drop_duplicates().sort_values("frame_idx").reset_index(drop=True)
_frame_vals = all_frames["frame_idx"].values
_idx = np.searchsorted(_stoppage_starts, _frame_vals, side="right") - 1
_valid = _idx >= 0
_in_range = np.zeros(len(_frame_vals), dtype=bool)
_idx_valid = _idx[_valid]
_in_range[_valid] = _frame_vals[_valid] <= _stoppage_ends[_idx_valid]
all_frames["is_stoppage"] = _in_range

stoppage_frame_lookup = all_frames.set_index("frame_idx")["is_stoppage"]

print(f"{len(stoppages)} stoppages >= {STOPPAGE_GAP_SEC}s flagged, covering {stoppages['n_frames'].sum()} frames "
      f"({stoppages['duration_sec'].sum():.1f}s total dead time)")

25 stoppages >= 10.0s flagged, covering 10901 frames (436.0s total dead time)


## 1. Proximity pressure — distance from each off-ball defender to the ball carrier
A defender only counts as "pressuring" if they're close AND (a) they're closing the distance fast (actively running at the carrier, >=2.5 m/s), or (b) they're within touch-tight range (<1.5m) regardless of speed, or (c) they're clearly ahead of their team's own out-of-possession baseline line (stepped out of the block). This is meant to separate real pressing from a deep, compact block that's simply already sitting near the ball because the attacker walked into it. Thresholds tightened from the first pass -- 1.0 m/s and 2.0m were too permissive and let most of a passive block still count as "pressuring".

In [4]:
PRESSURE_DIST_THRESH_M = 5.0
CLOSING_WINDOW_FRAMES = 12       # ~0.5s @ 25fps, window used to measure closing speed
CLOSING_SPEED_THRESH_MPS = 2.5   # must be closing at least this fast to count as an active press from open play (walking pace was too permissive)
TIGHT_MARK_DIST_M = 1.5          # inside this range, counts as pressure even without active closing (touch-tight)
STEP_OUT_MARGIN_M = 4.0          # must be at least this far ahead of the team's own average line depth

carrier_pos = ball_frame_table[
    ["frame_idx", "carrier_track_id", "carrier_team", "possession_team", "ball_pitch_x", "ball_pitch_y"]
]

pft = player_frame_table.merge(carrier_pos, on="frame_idx", how="left")

off_ball = pft[
    (pft["role"] == "player") &
    pft["carrier_track_id"].notna() &
    (pft["track_id"] != pft["carrier_track_id"]) &
    (pft["team"] != pft["carrier_team"])
].copy()

off_ball["dist_to_carrier"] = (
    (off_ball["pitch_x"] - off_ball["ball_pitch_x"]) ** 2 +
    (off_ball["pitch_y"] - off_ball["ball_pitch_y"]) ** 2
) ** 0.5

off_ball_pressure = off_ball.dropna(subset=["dist_to_carrier"]).copy()
off_ball_pressure = off_ball_pressure.sort_values(["track_id", "frame_idx"]).reset_index(drop=True)

# closing speed: how fast dist_to_carrier has been shrinking over the last CLOSING_WINDOW_FRAMES for this player
off_ball_pressure["dist_to_carrier_prev"] = off_ball_pressure.groupby("track_id")["dist_to_carrier"].shift(CLOSING_WINDOW_FRAMES)
off_ball_pressure["frame_gap"] = off_ball_pressure["frame_idx"] - off_ball_pressure.groupby("track_id")["frame_idx"].shift(CLOSING_WINDOW_FRAMES)
off_ball_pressure["closing_speed_mps"] = np.where(
    (off_ball_pressure["frame_gap"] > 0) & (off_ball_pressure["frame_gap"] <= CLOSING_WINDOW_FRAMES * 2),
    (off_ball_pressure["dist_to_carrier_prev"] - off_ball_pressure["dist_to_carrier"]) / (off_ball_pressure["frame_gap"] / FPS),
    np.nan
)

# stepped ahead of the team's own habitual defensive line -> not just standing in the block
off_ball_pressure["dist_from_own_goal"] = np.where(
    off_ball_pressure["team"] == 0,
    off_ball_pressure["pitch_x"],
    PITCH_LENGTH - off_ball_pressure["pitch_x"]
)
off_ball_pressure["dist_from_own_goal"] = off_ball_pressure["dist_from_own_goal"].clip(0, PITCH_LENGTH)
off_ball_pressure["team_line_avg"] = off_ball_pressure["team"].map(team_defensive_line_avg)
off_ball_pressure["stepped_out_of_line"] = (
    off_ball_pressure["dist_from_own_goal"] >= off_ball_pressure["team_line_avg"] + STEP_OUT_MARGIN_M
)

off_ball_pressure["is_pressuring"] = (
    (off_ball_pressure["dist_to_carrier"] < PRESSURE_DIST_THRESH_M) &
    (
        (off_ball_pressure["dist_to_carrier"] < TIGHT_MARK_DIST_M) |
        (off_ball_pressure["closing_speed_mps"].fillna(0) >= CLOSING_SPEED_THRESH_MPS) |
        off_ball_pressure["stepped_out_of_line"]
    )
)

team_pressure_per_frame = (
    off_ball_pressure.groupby(["frame_idx", "team", "carrier_team", "possession_team"], as_index=False)
    .agg(
        n_pressing=("is_pressuring", "sum"),
        n_defenders_on_pitch=("track_id", "nunique"),
        min_dist_to_carrier=("dist_to_carrier", "min"),
    )
)
team_pressure_per_frame["pressing_intensity"] = (
    team_pressure_per_frame["n_pressing"] / team_pressure_per_frame["n_defenders_on_pitch"]
)

## 2. Passing-lane pressure — how covered are the carrier's passing options

In [5]:
MIN_LANE_LENGTH_M = 3.0
TIGHT_LANE_THRESH_M = 1.5
CONTESTED_LANE_THRESH_M = 3.0

carrier_frames = pft[(pft["role"] == "player") & pft["carrier_track_id"].notna()].copy()

teammates = carrier_frames[
    (carrier_frames["team"] == carrier_frames["carrier_team"]) &
    (carrier_frames["track_id"] != carrier_frames["carrier_track_id"])
]
opponents = carrier_frames[carrier_frames["team"] != carrier_frames["carrier_team"]]

carrier_player_pos = (
    player_frame_table[player_frame_table["is_carrier"] == True]
    [["frame_idx", "track_id", "pitch_x", "pitch_y"]]
    .rename(columns={"track_id": "carrier_track_id", "pitch_x": "carrier_x", "pitch_y": "carrier_y"})
    .drop_duplicates(subset="frame_idx", keep="first")
)

teammate_pos = teammates[["frame_idx", "track_id", "pitch_x", "pitch_y"]].rename(
    columns={"track_id": "teammate_id", "pitch_x": "tm_x", "pitch_y": "tm_y"}
)
opponent_pos = opponents[["frame_idx", "track_id", "pitch_x", "pitch_y"]].rename(
    columns={"track_id": "opponent_id", "pitch_x": "opp_x", "pitch_y": "opp_y"}
)

lanes = teammate_pos.merge(opponent_pos, on="frame_idx", how="inner")
lanes = lanes.merge(carrier_player_pos, on="frame_idx", how="inner")
lanes = lanes.dropna(subset=["carrier_x", "carrier_y"])

Ax, Ay = lanes["carrier_x"].values, lanes["carrier_y"].values
Bx, By = lanes["tm_x"].values, lanes["tm_y"].values
Px, Py = lanes["opp_x"].values, lanes["opp_y"].values

ABx, ABy = Bx - Ax, By - Ay
APx, APy = Px - Ax, Py - Ay

seg_len_sq = ABx ** 2 + ABy ** 2
seg_len_sq = np.where(seg_len_sq == 0, 1e-9, seg_len_sq)

t = (APx * ABx + APy * ABy) / seg_len_sq
t_clipped = np.clip(t, 0.0, 1.0)

closest_x = Ax + t_clipped * ABx
closest_y = Ay + t_clipped * ABy
perp_dist = np.sqrt((Px - closest_x) ** 2 + (Py - closest_y) ** 2)

lanes["t"] = t
lanes["lane_perp_dist"] = perp_dist
lanes["lane_length_m"] = np.sqrt(seg_len_sq)

valid_lanes = lanes[
    (lanes["lane_length_m"] >= MIN_LANE_LENGTH_M) &
    (lanes["t"] >= 0.15) &
    (lanes["t"] <= 0.85)
].copy()

valid_lanes["lane_tight_block"] = valid_lanes["lane_perp_dist"] < TIGHT_LANE_THRESH_M
valid_lanes["lane_contested"] = valid_lanes["lane_perp_dist"] < CONTESTED_LANE_THRESH_M

lane_status = (
    valid_lanes.groupby(["frame_idx", "teammate_id"], as_index=False)
    .agg(lane_tight_block=("lane_tight_block", "any"), lane_contested=("lane_contested", "any"))
)

frame_lane_summary = (
    lane_status.groupby("frame_idx", as_index=False)
    .agg(
        n_lanes_total=("teammate_id", "nunique"),
        n_lanes_tight_blocked=("lane_tight_block", "sum"),
        n_lanes_contested=("lane_contested", "sum"),
    )
)
frame_lane_summary["n_lanes_open"] = frame_lane_summary["n_lanes_total"] - frame_lane_summary["n_lanes_contested"]
frame_lane_summary["frac_tight_blocked"] = frame_lane_summary["n_lanes_tight_blocked"] / frame_lane_summary["n_lanes_total"]
frame_lane_summary["frac_contested"] = frame_lane_summary["n_lanes_contested"] / frame_lane_summary["n_lanes_total"]

frame_lane_summary = frame_lane_summary.merge(
    carrier_pos[["frame_idx", "carrier_team", "possession_team"]], on="frame_idx", how="left"
)


## 3. Unified team pressure signal + events
Combines proximity pressure OR lane-blocking pressure, then excludes own-penalty-box frames (16.5m from own goal) since compact box defending is a different phenomenon from active pressing. This corrected `under_team_pressure` definition is used everywhere downstream (events, zone breakdown, time-to-press, pass resistance, weighted intensity).
Note: `n_pressing`/`pressing_intensity` now come from the closing-speed + step-out-of-line gated definition in section 1, not raw distance -- so a passive deep block no longer inflates this on its own. The lane-blocking signal (section 2) is left distance-based for now, since being within `TIGHT_LANE_THRESH_M` of an actual pass lane is a real obstruction regardless of how the defender got there.

In [6]:
N_PRESSING_THRESH = 2
FRAC_TIGHT_BLOCKED_THRESH = 0.375
MIN_EVENT_DURATION_SEC = 0.5
GAP_TOLERANCE_FRAMES = 5
BOX_DEPTH_M = 16.5

team_pressure_full = team_pressure_per_frame.merge(
    frame_lane_summary[["frame_idx", "n_lanes_total", "n_lanes_tight_blocked",
                         "n_lanes_contested", "frac_tight_blocked", "frac_contested"]],
    on="frame_idx",
    how="left"
)

ball_pos_by_frame = ball_frame_table[["frame_idx", "ball_pitch_x", "ball_pitch_y"]].rename(
    columns={"ball_pitch_x": "press_x", "ball_pitch_y": "press_y"}
)
team_pressure_full = team_pressure_full.merge(ball_pos_by_frame, on="frame_idx", how="left")

tpf = team_pressure_full.copy()
tpf["frac_tight_blocked"] = tpf["frac_tight_blocked"].fillna(0)

tpf["in_own_box"] = np.where(
    tpf["team"] == 0,
    tpf["press_x"] <= BOX_DEPTH_M,
    tpf["press_x"] >= PITCH_LENGTH - BOX_DEPTH_M
)

# lane blocking alone no longer triggers the flag: it must come with at least one defender
# actually within pressing distance of the carrier (otherwise a compact block near the box,
# with n_pressing == 0, was being counted as "pressure")
_lane_pressure = (
    (tpf["frac_tight_blocked"] >= FRAC_TIGHT_BLOCKED_THRESH) &
    (tpf["min_dist_to_carrier"] < PRESSURE_DIST_THRESH_M)
)
tpf["under_team_pressure"] = (
    ((tpf["n_pressing"] >= N_PRESSING_THRESH) | _lane_pressure) &
    (~tpf["in_own_box"])
)

# carry the corrected flag back onto team_pressure_full so every downstream section uses the same definition
team_pressure_full["under_team_pressure"] = tpf["under_team_pressure"]

tpf = tpf.sort_values(["team", "frame_idx"]).reset_index(drop=True)

events_list = []
for team_id, grp in tpf.groupby("team"):
    grp = grp.sort_values("frame_idx").reset_index(drop=True)

    is_press = grp["under_team_pressure"].values.copy()
    frames = grp["frame_idx"].values
    i, n = 0, len(is_press)
    while i < n:
        if not is_press[i]:
            j = i
            while j < n and not is_press[j]:
                j += 1
            gap_frame_span = frames[j - 1] - frames[i] + 1 if j > i else 0
            if i > 0 and j < n and gap_frame_span <= GAP_TOLERANCE_FRAMES:
                is_press[i:j] = True
            i = j
        else:
            i += 1
    grp["under_team_pressure_bridged"] = is_press

    grp["state_change"] = (grp["under_team_pressure_bridged"] != grp["under_team_pressure_bridged"].shift()).cumsum()
    runs = (
        grp.groupby("state_change")
        .agg(
            start_frame=("frame_idx", "first"),
            end_frame=("frame_idx", "last"),
            n_frames=("frame_idx", "count"),
            under_pressure=("under_team_pressure_bridged", "first"),
            possession_team=("possession_team", "first"),
            mean_n_pressing=("n_pressing", "mean"),
            mean_frac_tight_blocked=("frac_tight_blocked", "mean"),
        )
        .reset_index(drop=True)
    )
    runs["team"] = team_id
    events_list.append(runs)

team_pressure_events = pd.concat(events_list, ignore_index=True)
team_pressure_events = team_pressure_events[team_pressure_events["under_pressure"]].copy()
team_pressure_events["duration_sec"] = team_pressure_events["n_frames"] / FPS
team_pressure_events = team_pressure_events[
    team_pressure_events["duration_sec"] >= MIN_EVENT_DURATION_SEC
].reset_index(drop=True)

print("total team pressure events:", len(team_pressure_events))
print(team_pressure_events.groupby("team")["duration_sec"].describe())


total team pressure events: 166
      count      mean       std   min   25%   50%   75%   max
team                                                         
0.0    66.0  1.405455  1.024930  0.52  0.65  1.04  1.88  6.20
1.0   100.0  1.227200  0.744692  0.52  0.68  1.04  1.41  4.32


## 4. Total pressure time, normalized by each team's out-of-possession window

In [7]:
total_pressure_frames_by_team = team_pressure_events.groupby("team")["n_frames"].sum()
total_pressure_seconds_by_team = total_pressure_frames_by_team / FPS

bft_live = ball_frame_table.merge(stoppage_frame_lookup.rename("is_stoppage"), left_on="frame_idx", right_index=True, how="left")
bft_live["is_stoppage"] = bft_live["is_stoppage"].fillna(False)
bft_live = bft_live[~bft_live["is_stoppage"]]  # dead time isn't real out-of-possession play, exclude it

possession_frames_by_team = bft_live["possession_team"].value_counts(dropna=True)
total_labeled_frames = bft_live["possession_team"].notna().sum()
out_of_possession_frames = {
    0: total_labeled_frames - possession_frames_by_team.get(0, 0),
    1: total_labeled_frames - possession_frames_by_team.get(1, 0),
}
out_of_possession_seconds = {k: v / FPS for k, v in out_of_possession_frames.items()}

match_duration_sec = (player_frame_table["frame_idx"].max() + 1) / FPS

print("total pressure seconds by team:\n", total_pressure_seconds_by_team)
print(f"\nmatch duration: {match_duration_sec:.1f}s ({stoppages['duration_sec'].sum():.1f}s of that is stoppage time, excluded below)")
for team_id in [0, 1]:
    pct = total_pressure_seconds_by_team.get(team_id, 0) / out_of_possession_seconds[team_id] * 100
    print(f"team {team_id}: {total_pressure_seconds_by_team.get(team_id, 0):.1f}s pressing / "
          f"{out_of_possession_seconds[team_id]:.1f}s out of possession = {pct:.1f}% pressing intensity")

total pressure seconds by team:
 team
0.0     92.76
1.0    122.72
Name: n_frames, dtype: float64

match duration: 2813.0s (436.0s of that is stoppage time, excluded below)
team 0: 92.8s pressing / 897.0s out of possession = 10.3% pressing intensity
team 1: 122.7s pressing / 1274.1s out of possession = 9.6% pressing intensity


## Hand-off
Saves what the later notebooks need.

In [8]:
save_state(globals(), [
    'BOX_DEPTH_M',
    'CLOSING_SPEED_THRESH_MPS',
    'CLOSING_WINDOW_FRAMES',
    'FRAC_TIGHT_BLOCKED_THRESH',
    'LINE_HEIGHT_DEFENDERS_N',
    'N_PRESSING_THRESH',
    'PRESSURE_DIST_THRESH_M',
    'STEP_OUT_MARGIN_M',
    'STOPPAGE_GAP_SEC',
    'TIGHT_MARK_DIST_M',
    'ball_pos_by_frame',
    'frame_lane_summary',
    'lane_status',
    'match_duration_sec',
    'off_ball_pressure',
    'out_of_possession_seconds',
    'stoppage_frame_lookup',
    'stoppages',
    'team_pressure_events',
    'team_pressure_full',
    'total_pressure_seconds_by_team',
    'tpf',
], notebook='01_pressure_signal.ipynb')

saved 22 variables to C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\barca_atletico_first_half\new_cache\pressure_analysis\state
  BOX_DEPTH_M                       float           0.00 MB
  CLOSING_SPEED_THRESH_MPS          float           0.00 MB
  CLOSING_WINDOW_FRAMES             int             0.00 MB
  FRAC_TIGHT_BLOCKED_THRESH         float           0.00 MB
  LINE_HEIGHT_DEFENDERS_N           int             0.00 MB
  N_PRESSING_THRESH                 int             0.00 MB
  PRESSURE_DIST_THRESH_M            float           0.00 MB
  STEP_OUT_MARGIN_M                 float           0.00 MB
  STOPPAGE_GAP_SEC                  float           0.00 MB
  TIGHT_MARK_DIST_M                 float           0.00 MB
  ball_pos_by_frame                 DataFrame       1.69 MB
  frame_lane_summary                DataFrame       2.46 MB
  lane_status                       DataFrame       5.23 MB
  match_duration_sec                float64         0.00 MB
  off_